# Lecture 23 — Backpropagation: Sending the Error Backward

Derive the gradients by hand, then verify them with PyTorch.

In [ ]:
import numpy as np

## 1. Linear neuron

$$\hat y=wx+b,\qquad L=(\hat y-y)^2$$

In [ ]:
x, w, b, y = 2.0, 3.0, 1.0, 10.0
pred = w*x + b
loss = (pred - y)**2
dL_dw = 2*(pred-y)*x
dL_db = 2*(pred-y)
print('prediction:', pred)
print('loss:', loss)
print('dL/dw:', dL_dw)
print('dL/db:', dL_db)

## 2. One gradient-descent step

In [ ]:
lr = 0.1
w_new = w - lr*dL_dw
b_new = b - lr*dL_db
pred_new = w_new*x + b_new
loss_new = (pred_new-y)**2
print('new w:', w_new)
print('new b:', b_new)
print('new prediction:', pred_new)
print('new loss:', loss_new)

## 3. Finite-difference check

A numerical gradient can be estimated by:

$$f'(w)\approx\frac{f(w+\epsilon)-f(w-\epsilon)}{2\epsilon}$$

In [ ]:
eps = 1e-5
def loss_for_w(weight):
    p = weight*x + b
    return (p-y)**2

numerical = (loss_for_w(w+eps)-loss_for_w(w-eps))/(2*eps)
print('analytical:', dL_dw)
print('numerical :', numerical)

## 4. Hidden ReLU unit

$$z_1=w_1x+b_1,\quad h=ReLU(z_1),\quad \hat y=w_2h+b_2$$

In [ ]:
x = 2.0
w1, b1 = 2.0, 1.0
w2, b2 = 3.0, 0.0
target = 18.0

z1 = w1*x + b1
h = max(0.0, z1)
pred = w2*h + b2
loss = (pred-target)**2

dL_dpred = 2*(pred-target)
dpred_dh = w2
dh_dz1 = 1.0 if z1 > 0 else 0.0
dz1_dw1 = x
dL_dw1 = dL_dpred * dpred_dh * dh_dz1 * dz1_dw1

print('z1:', z1)
print('h:', h)
print('prediction:', pred)
print('loss:', loss)
print('dL/dw1:', dL_dw1)

## 5. Verify with PyTorch autograd

In [ ]:
import torch

x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(10.0)

pred = w*x + b
loss = (pred-y)**2
loss.backward()

print('prediction:', pred.item())
print('loss:', loss.item())
print('dL/dw:', w.grad.item())
print('dL/db:', b.grad.item())

## 6. Gradient flow challenge

Try replacing ReLU with sigmoid. Build a chain of 10 sigmoid layers with weights near 1 and inspect how the gradient changes. Then repeat with ReLU.

Question: why can multiplying many derivatives smaller than 1 make an early gradient tiny?

In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

z = 0.0
g = 1.0
for _ in range(10):
    s = sigmoid(z)
    g *= s*(1-s)
print('product of 10 sigmoid derivatives at z=0:', g)